# LeIsaac SO-101 PickOrange VLA Play

在 Isaac Sim 里跑 SO-101 单臂从厨房桌面抓橙子放盘子，对比多条 VLA 推理链路。

| 子章节 | 模型 | server | port | 状态 |
| --- | --- | --- | --- | --- |
| §1 | `hi-space/GR00T-N1.6-3B-Pick-Orange` | GR00T N1.6 (`run_gr00t_server.py`) | 5555 | ✅ 实测成功 |
| §2 | `LightwheelAI/leisaac-pick-orange-v0` | GR00T N1.5 (`inference_service.py`) | 5555 | ✅ 实测 1/1 |
| §3 | `shadowHokage/act_policy`, `edge-inference/smolvla-so101-pick-orange` | LeRobot async (`policy_server`) | 8080 | ✅ ACT 实测成功；SmolVLA 通推理链路 |
| §4 | `hi-space/GR00T-N1.7-3B-Pick-Orange` | GR00T N1.7 | 5555 | ⛔ ckpt 已就位，推理 infra 未搭 |

**前提条件**：
1. 在 `isaaclab-experience/` 目录下启动 jupyter
2. 启动前已激活 conda 环境：`conda activate isaaclab`
3. 已 `bash scripts/apply_leisaac_patches.sh`（如果首次拉 submodule）
4. 每个推理 cell 需要 Isaac Sim 弹窗，确保 `DISPLAY` 可用

**端口冲突注意**：§1/§2/§4 共用 :5555（GR00T 系列），任一时刻只能起一个。§3 的 LeRobot server (:8080) 与 GR00T 互不干扰，可共存。

## 0) 预检查

确保 LeIsaac submodule 文件齐全。

In [ ]:
!cd LeIsaac && test -f scripts/evaluation/policy_inference.py && test -f assets/robots/so101_follower.usd && test -f assets/scenes/kitchen_with_orange/scene.usd && python scripts/evaluation/policy_inference.py --help | head -n 20

## 1) GR00T N1.6 fine-tune (hi-space)

**模型**：`hi-space/GR00T-N1.6-3B-Pick-Orange` —— 社区在 LeIsaac PickOrange 数据上对 `nvidia/GR00T-N1.6-3B` 进行 fine-tune。N1.6 用 flow-matching action head，相比 N1.5 的 DiT diffusion 更新一代。

⚠ 与 §1.1 N1.5 共用 ZMQ :5555，先确保 N1.5 server 已停（`!bash scripts/policy_server.sh stop gr00t-n15`）。

### 1.1) 一键下载 fine-tuned ckpt（HF cache 已有则跳过）

In [ ]:
!bash scripts/download_hf_model.sh hi-space/GR00T-N1.6-3B-Pick-Orange

### 1.2) 一键启动 GR00T N1.6 推理服务（ZMQ :5555）

走 `server/start_server.sh --gr00t-only`，自动注入 `GR00T_MODEL_PATH` 和 `GR00T_EMBODIMENT_TAG=NEW_EMBODIMENT`。

In [ ]:
!bash scripts/policy_server.sh start gr00t-n16

### 1.3) 运行 SO-101 PickOrange 仿真实时推理

`--policy_type=gr00tn1.6` 走 `Gr00t16ServicePolicyClient`。

In [ ]:
!cd LeIsaac && PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=gr00tn1.6 --policy_host=127.0.0.1 --policy_port=5555 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_action_horizon=16 --device=cuda --enable_cameras

### 1.4) 停止 GR00T N1.6 推理服务

In [ ]:
!bash scripts/policy_server.sh stop gr00t-n16

## 2) GR00T N1.5 + LightwheelAI fine-tune

NVIDIA Isaac-GR00T N1.5（3B 参数，DiT diffusion action head）+ LightwheelAI 在仿真录制数据上 fine-tune 的 `leisaac-pick-orange-v0` checkpoint。

实测：1/1 成功率，机器人完成 pick-and-place 完整动作。

### 2.1) 一键下载 fine-tuned ckpt（~7.4 GB，已有则跳过）

In [ ]:
!bash scripts/download_hf_model.sh LightwheelAI/leisaac-pick-orange-v0


### 2.2) 一键启动 GR00T N1.5 推理服务（ZMQ :5555）

冷启动需加载 Eagle backbone + DiT head，约 20–30s；已启动则 idempotent skip。


In [ ]:
!bash scripts/policy_server.sh start gr00t-n15


### 2.3) 运行 SO-101 PickOrange 仿真实时推理

Isaac Sim 会弹窗显示 SO-101 单臂；客户端连 `:5555` 调 GR00T N1.5 推理。`-u` 是为了让 Episode/success 日志实时刷出来（Isaac Sim 退出会跳过 stdout flush）。


In [ ]:
!cd LeIsaac && PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=gr00tn1.5 --policy_host=127.0.0.1 --policy_port=5555 --policy_timeout_ms=15000 --policy_language_instruction='Pick up the orange and place it on the plate' --policy_action_horizon=16 --device=cuda --enable_cameras


### 2.4) 停止 GR00T N1.5 推理服务（释放显存）

In [ ]:
!bash scripts/policy_server.sh stop gr00t-n15


## 3) LeRobot async-inference (SmolVLA / ACT / 任意 LeRobot policy)

LeRobot v0.4+ 的 `policy_server` 是**通用 server**：client 通过 `RemotePolicyConfig` 告诉它该加载哪个 ckpt，所以一个 :8080 server 进程能同时服务 SmolVLA、ACT、pi0… 任何 LeRobot 框架的 policy。

区分发生在 client 端：`--policy_type=lerobot-<model_type>` + `--policy_checkpoint_path=<repo_id>`。下面演示两个 client：

| Client | model_type | ckpt | 输入 |
| --- | --- | --- | --- |
| SmolVLA | `lerobot-smolvla` | `edge-inference/smolvla-so101-pick-orange` | VLM（语言+视觉） + Action Expert，~450M |
| ACT | `lerobot-act` | `shadowHokage/act_policy` | 纯 vision + state → action chunk，~80M |

### 3.1) 安装 LeIsaac LeRobot client 依赖（首次执行后可跳过）

In [ ]:
!cd LeIsaac && pip install -e "source/leisaac[lerobot-async]"

### 3.2) 一键启动 LeRobot 推理服务（端口 :8080）

Idempotent：已起则跳过。SmolVLA 和 ACT 都连这一个 server。

In [ ]:
!bash scripts/policy_server.sh start lerobot

### 3.3) ACT policy client

`shadowHokage/act_policy` —— Action Chunking Transformer，LeRobot 框架下的轻量 imitation learning baseline（~80M 参数，纯 vision + state → action chunk）。其 `input_features` 直接用 `observation.images.front` / `observation.images.wrist`，与 LeIsaac sim 摄像头键名一致，无需 rename。

In [ ]:
!bash scripts/download_hf_model.sh shadowHokage/act_policy

In [ ]:
!cd LeIsaac && PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=lerobot-act --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_checkpoint_path=shadowHokage/act_policy --policy_action_horizon=16 --device=cuda --enable_cameras

### 3.4) SmolVLA fine-tune client

`edge-inference/smolvla-so101-pick-orange` —— 社区在 LightwheelAI/leisaac-pick-orange 数据集上对 `lerobot/smolvla_base` 进行 SO-101 PickOrange fine-tune。

In [ ]:
!bash scripts/download_hf_model.sh edge-inference/smolvla-so101-pick-orange

In [ ]:
!cd LeIsaac && PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 --policy_type=lerobot-smolvla --policy_host=127.0.0.1 --policy_port=8080 --policy_timeout_ms=15000 --policy_language_instruction='Pick the orange to the plate' --policy_checkpoint_path=edge-inference/smolvla-so101-pick-orange --policy_action_horizon=16 --device=cuda --enable_cameras

### 3.5) 停止 LeRobot 推理服务（释放显存）

不影响 GR00T `:5555`。

In [ ]:
!bash scripts/policy_server.sh stop lerobot

## 4) GR00T N1.7 fine-tune (hi-space) ⛔ infra 待搭

**模型**：`hi-space/GR00T-N1.7-3B-Pick-Orange` —— 上游 NVIDIA `Isaac-GR00T@n1.7-release`（commit `23ace64`）发布 N1.7 后，社区在 LeIsaac PickOrange 数据上的 fine-tune。

**当前状态**：ckpt 已下载到 HF cache（24GB，含 optimizer state，纯推理只需 12.6GB 的 3 个 safetensors）。但**推理端没法直接跑**，缺三件套：

1. **Isaac-GR00T-N1.7 仓库**：本地 `~/work/Isaac-GR00T` 还在 N1.6 commit `7d5a455`，不识别 `model_type: Gr00tN1d7`。需要：
   ```bash
   cd ~/work && git clone https://github.com/NVIDIA/Isaac-GR00T Isaac-GR00T-N1.7
   cd ~/work/Isaac-GR00T-N1.7 && git checkout n1.7-release
   ```
2. **新 conda env `gr00t-n17`**：torch / transformers / flash-attn 版本组合大概率和 N1.5/N1.6 不同。参考 `Isaac-GR00T-N1.7/pyproject.toml` 装一遍。
3. **LeIsaac client 端补丁**：
   - `source/leisaac/leisaac/policy/service_policy_clients.py` 加 `Gr00t17ServicePolicyClient`（参照 `Gr00t16ServicePolicyClient` 改）
   - `scripts/evaluation/policy_inference.py` 加 `gr00tn1.7` 分支
   - `scripts/policy_server.sh` 加 `gr00t-n17` backend，调用新仓库的 `gr00t/eval/run_gr00t_server.py`

上面三步搞完，下面 4.1~4.4 cell 即可填上对应命令。现在保留为占位骨架。

### 4.1) 一键下载 fine-tuned ckpt（HF cache 已有则跳过）

In [ ]:
!bash scripts/download_hf_model.sh hi-space/GR00T-N1.7-3B-Pick-Orange

### 4.2) 一键启动 GR00T N1.7 推理服务（ZMQ :5555） ⛔ 暂未实现

```bash
# 计划：
# !bash scripts/policy_server.sh start gr00t-n17
```

### 4.3) 运行 SO-101 PickOrange 仿真实时推理 ⛔ 暂未实现

```bash
# 计划：
# !cd LeIsaac && PYTHONUNBUFFERED=1 python -u scripts/evaluation/policy_inference.py \
#     --task=LeIsaac-SO101-PickOrange-v0 --eval_rounds=1 --episode_length_s=120 \
#     --policy_type=gr00tn1.7 --policy_host=127.0.0.1 --policy_port=5555 \
#     --policy_action_horizon=16 --device=cuda --enable_cameras
```

### 4.4) 停止 GR00T N1.7 推理服务 ⛔ 暂未实现